exportação do classificador para onnx

In [1]:
%pip install scikit-learn skl2onnx onnxruntime onnx joblib --quiet

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

warnings.filterwarnings('ignore')

CATEGORIAS = ["pro-ed", "pro-recovery"]

STOPWORDS_PT = frozenset("""
a ao aos aquela aquelas aquele aqueles aquilo as até com como contra
da das de dela delas dele deles depois do dos e ela elas ele eles em
entre era eram essa essas esse esses esta estamos estão estas estava
estavam este esteja estejam estes esteve estivemos estiver estiveram
estivesse estivessem estivermos estivéssemos estou está estávamos eu
fomos for fora foram fosse fossem fôssemos fui há isso isto já lá mas
me mesmo meu meus minha minhas mais menos na não nas nem no nos nós
nossa nossas nosso nossos num numa o os ou para pela pelas pelo pelos
por quando que quem são se seja sejam sem serei será serão seria seriam
seríamos seu seus só também te tem temos tenham tendo ter teu teus ti
tive tivemos tiver tiveram tivesse tivessem tivéssemos tu tua tuas um
uma você vocês vos à às é és
""".split())
print('dependências carregadas.')

dependências carregadas.


carregamento dos dados classificados

In [3]:
DATA_DIR = os.getcwd()
csv_path = os.path.join(DATA_DIR, "tweets_classificacao.csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"Arquivo não encontrado: {csv_path}\n"
        "Execute o notebook 8-classificador.ipynb primeiro para gerar o tweets_classificacao.csv."
    )

df = pd.read_csv(csv_path)
df = df.dropna(subset=["categoria"]).reset_index(drop=True)
n_total = len(df)
df = df[df["categoria"].isin(CATEGORIAS)].reset_index(drop=True)

print(f"Tweets rotulados: {n_total}  |  binários (pro-ed vs pro-recovery): {len(df)}")
print("\nDistribuição:")
print(df["categoria"].value_counts().reindex(CATEGORIAS, fill_value=0))

Tweets rotulados: 149  |  binários (pro-ed vs pro-recovery): 65

Distribuição:
categoria
pro-ed          61
pro-recovery     4
Name: count, dtype: int64


treinamento do modelo final (todos os dados)

In [4]:
SEED = 42
X = df["text_redacted"].astype(str)
y = df["categoria"]

vetorizador = TfidfVectorizer(
    lowercase=True, ngram_range=(1, 2), min_df=2, stop_words=list(STOPWORDS_PT)
)
X_tfidf = vetorizador.fit_transform(X)
n_features = X_tfidf.shape[1]

modelo = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
modelo.fit(X_tfidf, y)

print(f"treino: {len(df)} tweets | vocabulário tf-idf: {n_features} features (unigramas + bigramas)")
print(f"classes: {list(modelo.classes_)}")
print(f"acurácia de treino (referência): {modelo.score(X_tfidf, y):.3f}")

treino: 65 tweets | vocabulário tf-idf: 101 features (unigramas + bigramas)
classes: ['pro-ed', 'pro-recovery']
acurácia de treino (referência): 0.969


exportação para onnx

o onnx exporta apenas o classificador (LogisticRegression) sobre o vetor tf-idf; o vetorizador é salvo à parte via joblib. motivo: a operação TfIdfVectorizer do onnxruntime diverge da tokenização do sklearn (e não suporta bigramas), produzindo previsões incorretas; exportar só o classificador linear garante paridade exata com o sklearn (diff ~1e-8). a inferência combina os dois: texto -> vetorizador.transform -> modelo.onnx -> rótulo + probabilidades.

In [5]:
vec_path = os.path.join(DATA_DIR, "vetorizador_tfidf.joblib")
joblib.dump(vetorizador, vec_path)
print(f"vetorizador tf-idf salvo: {vec_path}")

initial_type = [("input", FloatTensorType([None, n_features]))]
onnx_model = convert_sklearn(modelo, initial_types=initial_type, target_opset=15)
onnx_path = os.path.join(DATA_DIR, "classificador_proed_prorecovery.onnx")
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())
print(f"modelo onnx salvo: {onnx_path}")

vetorizador tf-idf salvo: /Users/windisch/projects/edtwt/notebooks/vetorizador_tfidf.joblib
modelo onnx salvo: /Users/windisch/projects/edtwt/notebooks/classificador_proed_prorecovery.onnx


verificação: onnx vs sklearn

In [6]:
sess = rt.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
print("entradas:", [(i.name, list(i.shape), i.type) for i in sess.get_inputs()])
print("saídas:  ", [(o.name, o.type) for o in sess.get_outputs()])

amostra = X.iloc[:5].tolist()
X_amostra = vetorizador.transform(amostra).toarray().astype(np.float32)
out = sess.run(None, {sess.get_inputs()[0].name: X_amostra})
onx_label = list(out[0].ravel())
onx_proba = np.array([[d[c] for c in modelo.classes_] for d in out[1]])

sk_label = list(modelo.predict(vetorizador.transform(amostra)))
sk_proba = modelo.predict_proba(vetorizador.transform(amostra))

print("\nrótulos onnx:   ", onx_label)
print("rótulos sklearn:", sk_label)
print("rótulos iguais: ", onx_label == sk_label)
print(f"diff máx. probabilidade: {np.max(np.abs(onx_proba - sk_proba)):.2e}")

entradas: [('input', [None, 101], 'tensor(float)')]
saídas:   [('output_label', 'tensor(string)'), ('output_probability', 'seq(map(string,tensor(float)))')]

rótulos onnx:    ['pro-ed', 'pro-ed', 'pro-recovery', 'pro-ed', 'pro-ed']
rótulos sklearn: ['pro-ed', 'pro-ed', 'pro-recovery', 'pro-ed', 'pro-ed']
rótulos iguais:  True
diff máx. probabilidade: 4.64e-08


inferência de exemplo (texto -> previsão)

In [7]:
def prever(textos):
    """Texto cru -> rótulo + probabilidades via vetorizador + modelo ONNX."""
    feats = vetorizador.transform(textos).toarray().astype(np.float32)
    out = sess.run(None, {sess.get_inputs()[0].name: feats})
    labels = list(out[0].ravel())
    probs = np.array([[d[c] for c in modelo.classes_] for d in out[1]])
    return labels, probs


exemplos = [
    "estou me recuperando do transtorno alimentar, o tratamento salvou minha vida",
    "jejum e dieta restritiva para atingir o peso meta, thinspo",
    "hoje comi tudo e me sinto péssima, quero voltar à rotina",
]
labels, probs = prever(exemplos)
for texto, label, prob in zip(exemplos, labels, probs):
    print(f'"{texto[:55]}..." -> {label}  (pro-ed={prob[0]:.2f}, pro-recovery={prob[1]:.2f})')

"estou me recuperando do transtorno alimentar, o tratame..." -> pro-ed  (pro-ed=0.71, pro-recovery=0.29)
"jejum e dieta restritiva para atingir o peso meta, thin..." -> pro-ed  (pro-ed=0.71, pro-recovery=0.29)
"hoje comi tudo e me sinto péssima, quero voltar à rotin..." -> pro-ed  (pro-ed=0.55, pro-recovery=0.45)
